# Secure Blockchain — BUG HUNT Edition
**Peer Code-Review Exercise for Year-4 Students — Google Colab Edition**

> **Warning.** This notebook contains **five deliberately-planted security bugs**. Your job, working in pairs, is to:
>
> 1. **Review the code** against the Week 1 and Week 2 security checklists from the main notebook.
> 2. **Write a failing test** for each bug you suspect — a test that passes on the correct code and fails on the buggy code.
> 3. **Rank each bug** by severity (CRITICAL / HIGH / MEDIUM / LOW) and write a 1-sentence exploit description.
>
> Time budget: **60–90 minutes**. Target: **find at least 4 of the 5 bugs**. Aim for zero false positives — claiming a bug that isn't real costs you marks.
>
> **Do not run cells blindly trying to crash things.** Read. Reason. Then write a targeted test.

---

### Grading rubric

| Outcome | Marks |
|---------|------:|
| Bug correctly identified (location + explanation) | 15 each |
| Working failing test for the bug | 5 each |
| Correct severity ranking | 2 each |
| False positive (bug claimed that isn't real) | −5 each |

Maximum: 110. A perfect score requires finding all 5 with tests, with no false positives.

---

### Submission template (fill in the final markdown cell)

```
BUG 1
  File / function / line: ...
  Description:            ...
  Severity:               ...
  Exploit sketch:         ...
  Test:                   (code in cell below)
```


## Setup

In [ ]:
!pip install -q ecdsa==0.19.0
print("dependencies installed")

In [ ]:
import hashlib
import json
import time
import secrets
import random                 # <-- note this import
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Tuple
from ecdsa import SigningKey, VerifyingKey, SECP256k1, BadSignatureError

---
# Week 1 — Blockchain

Review these two classes carefully. Compare against the Week 1 checklist:
- Does `compute_hash` include **every** mutable field?
- Is hash input deterministic (fixed byte width, big-endian, UTF-8)?
- Does `is_valid` re-check the PoW target?
- Is the genesis deterministic?
- Are inputs bounded?


In [ ]:
MAX_DATA_BYTES = 64 * 1024
MAX_U64        = 2**64 - 1

@dataclass
class Block:
    index: int
    timestamp: int
    data: str
    prev_hash: str
    nonce: int = 0
    hash: str = ""

    def compute_hash(self) -> str:
        """Deterministic SHA-256 over the header fields."""
        if not (0 <= self.index <= MAX_U64):       raise OverflowError("index")
        if not (0 <= self.timestamp <= MAX_U64):   raise OverflowError("timestamp")
        if not (0 <= self.nonce <= MAX_U64):       raise OverflowError("nonce")
        if len(self.data.encode("utf-8")) > MAX_DATA_BYTES:
            raise ValueError("data too large")

        h = hashlib.sha256()
        h.update(self.index.to_bytes(8, "big"))
        h.update(self.timestamp.to_bytes(8, "big"))
        h.update(self.prev_hash.encode("utf-8"))
        h.update(self.nonce.to_bytes(8, "big"))
        return h.hexdigest()

    def mine(self, difficulty: int) -> None:
        target = "0" * difficulty
        while True:
            self.hash = self.compute_hash()
            if self.hash.startswith(target):
                return
            if self.nonce >= MAX_U64:
                raise OverflowError("nonce exhausted")
            self.nonce += 1

In [ ]:
class ChainError(Exception): pass

class Blockchain:
    def __init__(self, difficulty: int = 4):
        self.difficulty = difficulty
        genesis = Block(
            index=0,
            timestamp=0,
            data="GENESIS",
            prev_hash="0" * 64,
        )
        genesis.mine(difficulty)
        self.chain: List[Block] = [genesis]

    def add(self, data: str) -> Block:
        prev = self.chain[-1]
        block = Block(
            index=prev.index + 1,
            timestamp=int(time.time()),
            data=data,
            prev_hash=prev.hash,
        )
        block.mine(self.difficulty)
        self.chain.append(block)
        return block

    def is_valid(self) -> Tuple[bool, str]:
        for i in range(1, len(self.chain)):
            cur, prev = self.chain[i], self.chain[i - 1]
            if cur.compute_hash() != cur.hash:
                return False, f"block {i}: hash mismatch (tampered)"
            if cur.prev_hash != prev.hash:
                return False, f"block {i}: broken link to previous block"
            if cur.timestamp < prev.timestamp:
                return False, f"block {i}: non-monotonic timestamp"
            if cur.index != prev.index + 1:
                return False, f"block {i}: index out of order"
        return True, ""

### Smoke test — does the chain mine and validate?

In [ ]:
chain = Blockchain(difficulty=4)
for payload in ["Alice pays Bob 10", "Bob pays Carol 3"]:
    blk = chain.add(payload)
    print(f"mined block {blk.index}  nonce={blk.nonce:>6}  hash={blk.hash[:16]}...")
ok, why = chain.is_valid()
print("chain valid?", ok, why)

### Week 1 investigation cells

Use these cells to write your tests. Each test should:
- build a small chain,
- perform some operation (tamper, inject, etc.),
- assert what the **correct** behaviour would be.

If your assertion fails, you've likely found a bug.

In [ ]:
# YOUR BUG HUNT TESTS FOR WEEK 1 GO HERE
# Example (not a real bug): is a chain with one block valid?
c = Blockchain(difficulty=4)
ok, why = c.is_valid()
assert ok, why
print("example test passed")

In [ ]:
# (Space for more tests)

In [ ]:
# (Space for more tests)

---
# Week 2 — Wallet & Transactions

Review these three classes against the Week 2 checklist:
- Is the RNG `secrets` / `os.urandom`, never seeded `random`?
- Does `signing_hash` exclude the signature itself?
- Is `from_addr` re-derived from the pubkey and compared?
- Is the nonce check strict and the nonce incremented?
- Is `amount > 0` enforced? Is `from != to` enforced?
- Are balances checked before mutation?


In [ ]:
ADDR_LEN = 40

# Global RNG for wallet key generation.
_wallet_rng = random.Random(42)

def _wallet_entropy(n: int) -> bytes:
    return bytes(_wallet_rng.randint(0, 255) for _ in range(n))

class Wallet:
    def __init__(self):
        self._signing_key: SigningKey = SigningKey.generate(
            curve=SECP256k1, entropy=_wallet_entropy
        )
        self.verifying_key: VerifyingKey = self._signing_key.verifying_key

    def public_key_bytes(self) -> bytes:
        return self.verifying_key.to_string("compressed")

    def address(self) -> str:
        return hashlib.sha256(self.public_key_bytes()).hexdigest()[:ADDR_LEN]

    def sign(self, msg: bytes) -> bytes:
        return self._signing_key.sign(msg, hashfunc=hashlib.sha256)

    def __repr__(self):
        return f"Wallet(addr={self.address()})"

In [ ]:
class TxError(Exception): pass

MAX_AMOUNT = 2**64 - 1

@dataclass
class Transaction:
    from_addr: str
    to_addr:   str
    amount:    int
    nonce:     int
    public_key: str = ""
    signature:  str = ""

    def signing_hash(self) -> bytes:
        if not (0 < self.amount <= MAX_AMOUNT):
            raise TxError("amount out of range")
        if not (0 <= self.nonce <= MAX_U64):
            raise TxError("nonce out of range")
        if self.from_addr == self.to_addr:
            raise TxError("self-transfer not allowed")
        h = hashlib.sha256()
        h.update(self.from_addr.encode("utf-8"))
        h.update(self.to_addr.encode("utf-8"))
        h.update(self.amount.to_bytes(8, "big"))
        h.update(self.nonce.to_bytes(8, "big"))
        return h.digest()

    def sign_with(self, wallet: "Wallet") -> None:
        if wallet.address() != self.from_addr:
            raise TxError("wallet does not own from_addr")
        self.public_key = wallet.public_key_bytes().hex()
        self.signature  = wallet.sign(self.signing_hash()).hex()

    def verify(self) -> bool:
        if not self.public_key or not self.signature:
            raise TxError("unsigned transaction")

        pk_bytes = bytes.fromhex(self.public_key)
        vk = VerifyingKey.from_string(pk_bytes, curve=SECP256k1)
        try:
            vk.verify(bytes.fromhex(self.signature),
                      self.signing_hash(),
                      hashfunc=hashlib.sha256)
            return True
        except BadSignatureError:
            return False

In [ ]:
class State:
    def __init__(self, genesis_balances: Optional[Dict[str, int]] = None):
        self.balances: Dict[str, int] = dict(genesis_balances or {})
        self.nonces:   Dict[str, int] = {}

    def apply(self, tx: Transaction) -> None:
        if not tx.verify():
            raise TxError("invalid signature")

        expected = self.nonces.get(tx.from_addr, 0)
        if tx.nonce != expected:
            raise TxError(f"bad nonce (expected {expected}, got {tx.nonce})")

        bal = self.balances.get(tx.from_addr, 0)
        if bal < tx.amount:
            raise TxError("insufficient balance")

        self.balances[tx.from_addr] = bal - tx.amount
        self.balances[tx.to_addr]   = self.balances.get(tx.to_addr, 0) + tx.amount

    def snapshot(self) -> Dict[str, int]:
        return dict(self.balances)

### Happy-path demo — looks fine, right?

In [ ]:
# Reset the wallet RNG so this cell is re-runnable
_wallet_rng.seed(42)

alice = Wallet()
bob   = Wallet()
print("Alice:", alice.address())
print("Bob  :", bob.address())

state = State(genesis_balances={alice.address(): 100})
tx = Transaction(alice.address(), bob.address(), 10, nonce=0)
tx.sign_with(alice)
print("signature valid?", tx.verify())
state.apply(tx)
print("balances:", state.snapshot())
print("looks fine... or does it?")

### Week 2 investigation cells

Write one test per suspected bug. Suggestions for what to probe:

- Create two wallets. Are their addresses **different** every run? Across sessions?
- Can `Mallory` construct a transaction that claims `from_addr = alice.address()` without holding Alice's key?
- What happens if you call `state.apply(tx)` twice with the same `tx`?
- What happens if you sign a tx, then modify a field, then verify?
- Does `is_valid` reject a hand-crafted block whose hash does not start with the required zeros?

In [ ]:
# YOUR BUG HUNT TESTS FOR WEEK 2 GO HERE
# Hint: start with the simplest possible invariant you can write down.


In [ ]:
# (Space for more tests)

In [ ]:
# (Space for more tests)

In [ ]:
# (Space for more tests)

---
## Your findings

Complete this template. One section per suspected bug. Keep descriptions short — one or two sentences. Your test cells above should *demonstrate* each bug; this section *explains* it.

### BUG 1
- **File / class / method**:
- **Description of the defect**:
- **Severity** (CRITICAL / HIGH / MEDIUM / LOW):
- **Exploit sketch** (one sentence — how does an attacker benefit?):
- **Test cell** (which cell number above demonstrates it?):

### BUG 2
- **File / class / method**:
- **Description**:
- **Severity**:
- **Exploit sketch**:
- **Test cell**:

### BUG 3
- **File / class / method**:
- **Description**:
- **Severity**:
- **Exploit sketch**:
- **Test cell**:

### BUG 4
- **File / class / method**:
- **Description**:
- **Severity**:
- **Exploit sketch**:
- **Test cell**:

### BUG 5
- **File / class / method**:
- **Description**:
- **Severity**:
- **Exploit sketch**:
- **Test cell**:

### (Optional) Other findings
Anything that looks wrong but you're not sure about? List it here. You won't be penalised for a hedged observation, only for a confidently-asserted false positive in the five main slots above.
